# Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from llm.interface.qwen import Qwen

In [ ]:
import sqlite3
import pandas as pd
from tqdm import tqdm
from llm.prompts import clear_schema_system_prompt
from utils import format_schema_with_samples, parse_code_string

In [ ]:
model = Qwen("llm/weight/qwen25-7b")
model.load_model()
model.load_tokenizer()

In [ ]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [ ]:
question = "Among the schools with the average score in Math over 560 in the SAT test, how many schools are in the bay area?"
target_schema = "['School ID', 'School Name', 'Average Math Score', 'Is in Bay Area']"

# Plan Generation

## Preparation

In [1]:
DATA_SRC = '../../data_src'
tables = {
    'Table_0': 'pandas_dfs/codebase_community/comments.csv',
    'Table_1': 'pandas_dfs/codebase_community/posts.csv',
    'Table_2': 'pandas_dfs/california_schools/satscores.csv',
}

In [6]:
x = [
        "CD Number",
        "School Type",
        "School Name",
        "District Name",
        "County Name",
        "Enrollment 12",
        "Number of Test Takers",
        "Average Score Reading",
        "Average Score Math",
        "Average Score Writing",
        "Number of GE 1500",
]
print(x)

['CD Number', 'School Type', 'School Name', 'District Name', 'County Name', 'Enrollment 12', 'Number of Test Takers', 'Average Score Reading', 'Average Score Math', 'Average Score Writing', 'Number of GE 1500']


In [5]:
import pandas as pd
df = pd.read_csv(f'{DATA_SRC}/{tables['Table_2']}')
df

,cds,rtype,sname,dname,cname,enroll12,NumTstTakr,AvgScrRead,AvgScrMath,AvgScrWrite,NumGE1500
0,1100170000000,D,NaN,Alameda County Office of Education,Alameda,398,88,418.0,418.0,417.0,14.0
1,1100170109835,S,FAME Public Charter,Alameda County Office of Education,Alameda,62,17,503.0,546.0,505.0,9.0
2,1100170112607,S,Envision Academy for Arts & Technology,Alameda County Office of Education,Alameda,75,71,397.0,387.0,395.0,5.0
3,1100170118489,S,Aspire California College Preparatory Academy,Alameda County Office of Education,Alameda,61,0,NaN,NaN,NaN,NaN
4,1611190000000,D,NaN,Alameda Unified,Alameda,922,544,521.0,546.0,519.0,333.0
...,...,...,...,...,...,...,...,...,...,...,...
2264,58727365830054,S,Lincoln (Abraham) (Alternative),Marysville Joint Unified,Yuba,97,0,NaN,NaN,NaN,NaN
2265,58727365830138,S,Marysville Charter Academy for the Arts,Marysville Joint Unified,Yuba,41,29,501.0,494.0,484.0,16.0
2266,58727365835202,S,Marysville High,Marysville Joint Unified,Yuba,197,53,489.0,513.0,487.0,24.0
2267,58727690000000,D,NaN,Wheatland Union High,Yuba,160,54,480.0,475.0,463.0,21.0


In [ ]:
.head()

In [ ]:
describe_table_system_prompt = """Assume you are the creator of a table.

Given the table represented by its schema and sample rows, briefly describe what the table likely represents."""

In [ ]:
updated_schemas: list[str] = []
for table in tqdm(tables):
    df = pd.read_csv(f'{DATA_SRC}/{tables[table]}')
    first_step_msg = [
        {'role': 'system', 'content': clear_schema_system_prompt},
        {'role': 'user', 'content': f'Table: {format_schema_with_samples(df)}'}
    ]
    output = model.chat(first_step_msg)
    updated_schema = parse_code_string(output)
    df.columns = updated_schema
    df.to_sql(table, conn, index=False, if_exists="replace")
    updated_schemas.append(updated_schema)

In [ ]:
for i in updated_schemas:
    print(i)

## Step 1: Get Base Table

In [ ]:
# updated_schemas = [
#     ["ID", "Post ID", "Score", "Text", "Creation Date", "User ID", "User Display Name"],
#     [
#         "ID",
#         "Post Type ID",
#         "Accepted Answer ID",
#         "Creation Date",
#         "Score",
#         "View Count",
#         "Body",
#         "Owner User ID",
#         "Last Activity Date",
#         "Title",
#         "Tags",
#         "Answer Count",
#         "Comment Count",
#         "Favorite Count",
#         "Last Editor User ID",
#         "Last Edit Date",
#         "Community Owned Date",
#         "Parent ID",
#         "Closed Date",
#         "Owner Display Name",
#         "Last Editor Display Name",
#     ],
#     [
#         "CD Number",
#         "School Type",
#         "School Name",
#         "District Name",
#         "County Name",
#         "Enrollment 12",
#         "Number of Test Takers",
#         "Average Score Reading",
#         "Average Score Math",
#         "Average Score Writing",
#         "Number of GE 1500",
#     ],
# ]

In [ ]:
from utils import format_schema
available_tables = ""
for table_idx, table in enumerate(tables):
    df = pd.read_csv(f'{DATA_SRC}/{tables[table]}', names=updated_schemas[table_idx], skiprows=1)
    available_tables += f"\n{table}: ```{format_schema(df)}```\n"

In [ ]:
from llm.prompts import plan_generator_first_step_system_prompt
first_step_msg = [
    {'role': 'system', 'content': plan_generator_first_step_system_prompt},
    {'role': 'user', 'content': f'Question: {question}\nAvailable Tables: {available_tables}\nTarget Schema: {target_schema}'}
]

In [ ]:
first_step_out = model.chat(first_step_msg)

In [ ]:
first_step_out = parse_code_string(first_step_out)

In [ ]:
print(first_step_out)

## Step 1.5: Extract Base Table from Memory

In [ ]:
# first_step_out = {
#     "operation": "select_table",
#     "tables_involved": ["Table_2"],
#     "description": "Select Table_2."
# }

In [ ]:
extract_base_sql = model.chat(
    [
        {
            "role": "user",
            "content": f"Convert this description: {first_step_out} to SQL code (sqlite3). Please answer directly; ensure that your output can be parsed directly as SQL code.",
        }
    ]
)

In [ ]:
print(extract_base_sql)

In [ ]:
base_table = pd.read_sql('SELECT * FROM Table_2;', conn)

In [ ]:
extract_column_table = base_table.to_sql('base_table', conn, index=False, if_exists="replace")

## Step 2: Column Projection

In [ ]:
col_projection_system_prompt = """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema that we will transform the source table into in a step-by-step manner.
- A column from the target schema as the current target column.

Your goal is to determine whether to select a certain column from SRC or extract information from certain column(s) from SRC to form the target column.

The output format for selecting a certain column:
{
    "operation": "select_column",
    "columns_involved": ["Restaurant ID"],
    "description": "Select SRC.Restaurant ID."
}

While for extracting information from certain column(s):
{
    "operation": "extract_column",
    "columns_involved": ["City", "ZIP Code"],
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`."
}

Output your result strictly as a Python dictionary, without any extra formatting, explanations, or text. The output must be directly parseable as a Python dictionary."""

In [ ]:
from ast import literal_eval
target_schema_parsed = literal_eval(target_schema)
second_step_msg = [
    {'role': 'system', 'content': col_projection_system_prompt},
    {'role': 'user', 'content': f'Source table: ```{format_schema_with_samples(base_table)}```\nTarget Schema: {target_schema_parsed}\nTarget Column: `{target_schema_parsed[2]}`'}
]

In [ ]:
second_step_output = model.chat(second_step_msg)

In [ ]:
print(second_step_output)

## Step 2a: Execute

In [ ]:
operations = [
    {
        "operation": "select_column",
        "columns_involved": ["CD Number"],
        "description": "Select SRC.CD Number.",
    },
    {
        "operation": "select_column",
        "columns_involved": ["School Name"],
        "description": "Select SRC.School Name.",
    },
    {
        "operation": "select_column",
        "columns_involved": ["Average Score Math"],
        "description": "Select SRC.`Average Score Math`.",
    },
    {
        "operation": "extract_column",
        "columns_involved": ["District Name", "County Name"],
        "description": "Check if SRC.District Name or SRC.County Name contains 'Bay Area'.",
    },
]

### Extract_Column Operation

In [1]:
import pandas as pd
import sqlite3
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
base_table = pd.read_csv('base_table.csv')
base_table.to_sql('base_table', conn, if_exists='replace', index=False)
extract_col_table = pd.read_sql('SELECT "District Name", "County Name" FROM base_table;', conn)

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from llm.interface.qwen import Qwen

In [4]:
model = Qwen("llm/weight/qwen25-7b")
model.load_model()
model.load_tokenizer()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [16]:
extract_col_system_prompt = """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table represented by its schema and rows.
- A column to be added to this table whose values depend on the other columns in the table.

Your goal is to determine the values of the new column for all rows. Ensure you consider **all provided columns together** rather than relying on a single column. For example, a city name may exist in multiple locations, but when paired with its corresponding province or county, ambiguity is reduced.

Output your result strictly as a Python list representing the new column values for all rows, without any extra formatting, explanations, or text. The output must be directly parseable as a Python list."""

In [ ]:
# extract_col_system_prompt = """You are a helpful and knowledgeable data scientist.

# You will be provided with:
# - A table represented by its schema and rows.
# - A column to be added to this table whose values depend on the other columns in the table.

# Your goal is to determine the values of the new column for all rows. Ensure you consider **all provided columns together** rather than relying on a single column. For example, a city name may exist in multiple locations, but when paired with its corresponding province or county, ambiguity is reduced.

# Output your result strictly as a Python list representing the new column values and very brief reasoning for all rows, without any extra formatting, explanations, or text. The output must be directly parseable as a Python list.

# Output format:
# [{'value': ..., 'reasoning': ...}, ...]"""

In [17]:
unique_ex_col_tbl = extract_col_table.drop_duplicates().reset_index(drop=True)

In [18]:
rows = []
inc = 5
for i in range(0, len(unique_ex_col_tbl), inc):
    rows.append((i, i+inc))
rows[-1] = (rows[-1][0], len(unique_ex_col_tbl))

In [19]:
from utils import format_schema_extensive, parse_code_string
from tqdm import tqdm
new_col_values = []
for row in tqdm(rows):
# for row in tqdm([(10, 11)]):
    extract_col_msg = [
        {'role': 'system', 'content': extract_col_system_prompt},
        {'role': 'user', 'content': f'Table ({row[1]-row[0]} rows): ```{format_schema_extensive(unique_ex_col_tbl, row[0], row[1])}```\nOverall Schema: {list(base_table.columns)}\nNew Column: `Is in Bay Area` (data type: boolean)'}
    ]
    extract_col_output = model.chat(extract_col_msg)
    new_col_values.extend(parse_code_string(extract_col_output))

100%|██████████| 105/105 [06:11<00:00,  3.54s/it]


In [20]:
results_cache = dict()
columns = unique_ex_col_tbl.columns
for idx, row in unique_ex_col_tbl.iterrows():
    vals = []
    for col in columns:
        vals.append(row[col])
    key = "_SEP_".join(vals)
    results_cache[key] = new_col_values[idx]

In [23]:
actual_values = []
for idx, row in extract_col_table.iterrows():
    vals = []
    for col in columns:
        vals.append(row[col])
    key = "_SEP_".join(vals)
    actual_values.append(results_cache[key])

In [27]:
y = base_table[["CD Number", "School Name", "Average Score Math"]]
y['Is in Bay Area'] = actual_values
y.to_csv('processor_output.csv', index=False)

/tmp/ipykernel_3859774/2257166756.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y['Is in Bay Area'] = actual_values


## FINAL STEP: Answer the Question

In [28]:
proc = pd.read_csv('processor_output.csv')

In [29]:
qa_sys_prompt = """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table represented by its schema and rows.
- A question over the table.

Your goal is to produce a SQL code (SQLite) to answer the question using the table.

Output your result strictly as a SQL code (SQLite) without any extra formatting, explanations, or text. The output must be directly parseable as a SQL code."""

In [30]:
question = "Among the schools with the average score in Math over 560 in the SAT test, how many schools are in the bay area?"

In [31]:
msg = [
    {'role': 'system', 'content': qa_sys_prompt},
    {'role': 'user', 'content': f'Table `proc`: ```{format_schema_extensive(proc, 0, 3)}```\nQuestion: {question}'},
]

In [32]:
model.chat(msg)

"```sql\nSELECT COUNT(*) \nFROM proc \nWHERE Average_Score_Math > 560 AND Is_in_Bay_Area = 'True';\n```"

In [35]:
proc.to_sql('proc', conn, index=False, if_exists="replace")

2269

In [41]:
result = pd.read_sql("""SELECT COUNT(*) FROM proc WHERE "Average Score Math" > 560 AND "Is in Bay Area" = True;""", conn)

In [42]:
result

,COUNT(*)
0,58
